# ML Requirements and Model Ladder

This notebook defines the first implementation requirements for scikit-learn, PyTorch, and TensorFlow models on the molecule-analysis datasets.

Goals:
- verify framework availability
- map datasets to task families
- define the first model ladder for each framework
- prepare a consistent evaluation contract

In [28]:
import os
from pathlib import Path

import pandas as pd
import sklearn
import tensorflow as tf
import torch
from IPython.display import display

2026-06-28 11:47:39.160529: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 11:47:39.744315: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-28 11:47:40.014305: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-28 11:47:40.016631: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-28 11:47:40.384527: I tensorflow/core/platform/cpu_feature_gua

In [29]:
PROJECT_ROOT = Path.cwd().parent.resolve()
DATA_DIR = Path(os.environ.get("DATA_DIR"))

versions = pd.DataFrame(
    {
        "framework": ["scikit-learn", "PyTorch", "TensorFlow"],
        "version": [sklearn.__version__, torch.__version__, tf.__version__],
    }
)
display(versions)

,framework,version
0,scikit-learn,1.8.0
1,PyTorch,2.2.2+cu121
2,TensorFlow,2.16.2


## Dataset-to-Task Mapping

In [30]:
dataset_plan = pd.DataFrame(
    [
        {
            "dataset": "BBBP.csv",
            "task": "binary classification",
            "tier": "A",
            "recommended_first_model": "char n-gram logistic regression",
        },
        {
            "dataset": "clintox.csv",
            "task": "binary or multitask classification",
            "tier": "A",
            "recommended_first_model": "binary logistic regression / small MLP",
        },
        {
            "dataset": "delaney-processed.csv",
            "task": "regression",
            "tier": "A",
            "recommended_first_model": "linear regression / tree regressor",
        },
        {
            "dataset": "Lipophilicity.csv",
            "task": "regression",
            "tier": "A",
            "recommended_first_model": "ridge regression / MLP regressor",
        },
        {
            "dataset": "SAMPL.csv",
            "task": "regression",
            "tier": "A",
            "recommended_first_model": "linear regression baseline",
        },
        {
            "dataset": "HIV.csv",
            "task": "binary classification",
            "tier": "B",
            "recommended_first_model": "logistic regression and calibrated tree models",
        },
        {
            "dataset": "tox21.csv",
            "task": "multitask classification with missing labels",
            "tier": "B",
            "recommended_first_model": "one-vs-rest or shared-head MLP",
        },
        {
            "dataset": "sider.csv",
            "task": "multilabel classification",
            "tier": "B",
            "recommended_first_model": "multilabel logistic or MLP",
        },
        {
            "dataset": "muv.csv",
            "task": "sparse multitask classification",
            "tier": "B",
            "recommended_first_model": "masked loss baseline",
        },
        {
            "dataset": "toxcast_data.csv",
            "task": "wide sparse multitask classification",
            "tier": "C",
            "recommended_first_model": "feature selection plus sparse baseline",
        },
        {
            "dataset": "qm8.csv",
            "task": "multi-target regression",
            "tier": "C",
            "recommended_first_model": "multi-output regressor after header cleanup",
        },
        {
            "dataset": "qm9.csv",
            "task": "multi-target regression",
            "tier": "C",
            "recommended_first_model": "multi-output regressor or MLP",
        },
    ]
)
display(dataset_plan)

,dataset,task,tier,recommended_first_model
0,BBBP.csv,binary classification,A,char n-gram logistic regression
1,clintox.csv,binary or multitask classification,A,binary logistic regression / small MLP
2,delaney-processed.csv,regression,A,linear regression / tree regressor
3,Lipophilicity.csv,regression,A,ridge regression / MLP regressor
4,SAMPL.csv,regression,A,linear regression baseline
5,HIV.csv,binary classification,B,logistic regression and calibrated tree models
6,tox21.csv,multitask classification with missing labels,B,one-vs-rest or shared-head MLP
7,sider.csv,multilabel classification,B,multilabel logistic or MLP
8,muv.csv,sparse multitask classification,B,masked loss baseline
9,toxcast_data.csv,wide sparse multitask classification,C,feature selection plus sparse baseline


## scikit-learn Requirements

Required concepts:
- feature matrices and target vectors
- train/validation/test split discipline
- pipelines and preprocessing
- classification and regression metrics
- model calibration and class imbalance awareness

Recommended first ladder:
1. logistic regression
2. random forest
3. gradient boosting
4. multi-output and multi-label wrappers

## PyTorch Requirements

Required concepts:
- tensors and shape discipline
- Dataset and DataLoader abstractions
- forward pass, loss, backward pass, optimizer step
- device placement and batching
- training history logging

Recommended first ladder:
1. linear regression
2. binary classification MLP
3. shared-trunk multitask MLP
4. graph neural network extension later

## TensorFlow Requirements

Required concepts:
- Keras Sequential and Functional APIs
- losses, metrics, optimizers, callbacks
- tf.data or array-based feeding
- validation monitoring and early stopping

Recommended first ladder:
1. dense regression model
2. dense binary classifier
3. multi-output dense model
4. graph extension later

## Evaluation Contract

In [31]:
evaluation_contract = pd.DataFrame(
    [
        {
            "task_type": "binary classification",
            "primary_metrics": "ROC AUC, F1, precision, recall, accuracy",
            "notes": "use stratified splits",
        },
        {
            "task_type": "multilabel or multitask classification",
            "primary_metrics": "macro F1, micro F1, AUROC per task, label coverage",
            "notes": "mask missing labels where needed",
        },
        {
            "task_type": "regression",
            "primary_metrics": "RMSE, MAE, R2",
            "notes": "inspect residuals and target scale",
        },
        {
            "task_type": "multi-target regression",
            "primary_metrics": "per-target RMSE and MAE, averaged summary",
            "notes": "watch target-scale differences",
        },
    ]
)
display(evaluation_contract)

,task_type,primary_metrics,notes
0,binary classification,"ROC AUC, F1, precision, recall, accuracy",use stratified splits
1,multilabel or multitask classification,"macro F1, micro F1, AUROC per task, label coverage",mask missing labels where needed
2,regression,"RMSE, MAE, R2",inspect residuals and target scale
3,multi-target regression,"per-target RMSE and MAE, averaged summary",watch target-scale differences


## First Implementation Milestone

The first end-to-end milestone should deliver:
- one scikit-learn classification notebook on BBBP
- one scikit-learn regression notebook on Delaney or Lipophilicity
- one PyTorch binary classifier mirroring BBBP
- one TensorFlow binary classifier or regressor mirroring the same data split
- one comparison notebook summarizing metrics and tradeoffs